# `casacoretables` — full Python API tour

[`casacoretables`](https://github.com/) is a standalone, minimal-dependency build of
casacore's **table system** with a pybind11 interface. Its API mirrors
`python-casacore`'s `casacore.tables` (minus the MeasurementSet helpers, which live in
casacore's `ms` module).

This notebook walks through the **entire public Python API**:

1. Building a table description and creating a table
2. Table metadata
3. Columns — `putcol` / `getcol` (and zero-copy)
4. Cells and cell slices
5. Column slices
6. Variable-shaped (array) columns
7. Keywords (table and column)
8. Adding / removing / renaming columns and rows
9. `tablecolumn` — the column accessor
10. `tablerow` — row-at-a-time access
11. TaQL queries (`taql`, `query`, `select`, `sort`, `calc`)
12. Iterating a table (`iter` / `tableiter`)
13. Indexed lookups (`index` / `tableindex`)
14. Copy / rename / delete, and ASCII import/export
15. Re-opening tables, sub-tables, and the table cache

Everything runs in a scratch directory so it is safe to execute top-to-bottom.

In [1]:
import os, tempfile, shutil
import numpy as np

# casacoretables top-level table API
from casacoretables.tables import (
    table, taql, tablecommand,
    maketabdesc, makescacoldesc, makearrcoldesc, makecoldesc, makedminfo,
    tableexists, tableiswritable, tableinfo,
    tablecopy, tablerename, tabledelete,
    tablefromascii, tabledefinehypercolumn,
)

# Work in a throwaway directory.
workdir = tempfile.mkdtemp(prefix="casacoretables_demo_")
os.chdir(workdir)
print("working in:", workdir)

working in: /var/folders/b7/dx896v1x4yjb9v6rvs_n2hs00000gp/T/casacoretables_demo_cogl044b


## 1. Build a table description and create a table

A table is described by a set of **column descriptors**:

* `makescacoldesc(name, value)` — a *scalar* column; the example `value` fixes the data type.
* `makearrcoldesc(name, value, ndim=, shape=)` — an *array* column (fixed or variable shape).

`maketabdesc([...])` bundles them into a table description, which `table(name, desc, nrow=...)`
turns into a new table on disk.

In [2]:
desc = maketabdesc([
    makescacoldesc("ANTENNA", 0),                       # int32 scalar
    makescacoldesc("NAME", ""),                         # string scalar
    makescacoldesc("TIME", 0.0),                        # float64 scalar
    makescacoldesc("FLAG_ROW", False),                  # bool scalar
    makearrcoldesc("UVW", 0.0, shape=[3]),              # fixed 1-D float64 array
    makearrcoldesc("DATA", 0.0 + 0.0j, ndim=2),         # variable 2-D complex128 array
])

t = table("demo.tab", desc, nrow=5, ack=False)
print("created:", t.name())
print("nrows =", t.nrows(), " ncols =", t.ncols())
print("columns:", t.colnames())

created: /private/var/folders/b7/dx896v1x4yjb9v6rvs_n2hs00000gp/T/casacoretables_demo_cogl044b/demo.tab
nrows = 5  ncols = 6
columns: ['ANTENNA', 'NAME', 'TIME', 'FLAG_ROW', 'UVW', 'DATA']


## 2. Table metadata

`tableexists` / `tableiswritable` / `tableinfo` are module-level helpers; the `table`
object also exposes `info()`, `putinfo()`, `endianformat()`, `iswritable()`, `getdesc()`,
`getdminfo()`, and a human-readable `__str__`.

In [3]:
print("exists  :", tableexists("demo.tab"))
print("writable:", tableiswritable("demo.tab"))

t.putinfo({"type": "demo", "subType": "tour"})
t.addreadmeline("created by the casacoretables API tour")
print("info    :", t.info())
print("endian  :", t.endianformat())
print("col types:", {c: t.coldatatype(c) for c in t.colnames()})

exists  : True
writable: True
info    : {'type': 'demo', 'subType': 'tour', 'readme': 'created by the casacoretables API tour\n'}
endian  : little
col types: {'ANTENNA': 'int', 'NAME': 'string', 'TIME': 'double', 'FLAG_ROW': 'boolean', 'UVW': 'double', 'DATA': 'dcomplex'}


## 3. Columns: `putcol` / `getcol`

`putcol`/`getcol` move whole columns as numpy arrays. The first axis is the row axis;
any remaining axes are the cell shape (casacore's column-major storage is presented in
numpy's row-major convention automatically).

In [4]:
t.putcol("ANTENNA", np.arange(5, dtype=np.int32))
t.putcol("NAME", ["ea01", "ea02", "ea03", "ea04", "ea05"])
t.putcol("TIME", np.linspace(0.0, 4.0, 5))
t.putcol("UVW", np.arange(5 * 3, dtype=float).reshape(5, 3))

print("ANTENNA:", t.getcol("ANTENNA"))
print("NAME   :", t.getcol("NAME"))
print("UVW shape:", t.getcol("UVW").shape)   # (nrow, 3)

ANTENNA: [0 1 2 3 4]
NAME   : ['ea01', 'ea02', 'ea03', 'ea04', 'ea05']
UVW shape: (5, 3)


### Zero-copy reads

`getcol` returns a numpy **view** over casacore's storage (no copy): the array does not
own its data (`OWNDATA == False`) and its `base` is a capsule that keeps the underlying
C++ buffer alive. For an explicit fill into a pre-allocated array use `getcolnp`.

In [5]:
uvw = t.getcol("UVW")
print("OWNDATA:", uvw.flags["OWNDATA"], "| base:", type(uvw.base).__name__)

# Explicit zero-copy fill into a caller-provided array:
out = np.empty((5, 3), dtype=float)
t.getcolnp("UVW", out)
print("getcolnp filled:", np.array_equal(out, uvw))

OWNDATA: False | base: PyCapsule
getcolnp filled: True


## 4. Cells and cell slices

`getcell` / `putcell` operate on one row of one column. For array cells you can read or
write a sub-region with `getcellslice` / `putcellslice` (`blc`/`trc` are inclusive
bottom-left / top-right corners, in numpy axis order).

In [6]:
# DATA is a variable-shaped 2-D complex column: give each row its own shape.
for row in range(5):
    t.putcell("DATA", row, np.full((2, 4), row + 1j, dtype=np.complex128))

print("cell DATA[2] shape:", t.getcell("DATA", 2).shape)

# A slice of one cell: rows 0..0, cols 1..2 of DATA[2]
sl = t.getcellslice("DATA", 2, blc=[0, 1], trc=[0, 2])
print("DATA[2][0,1:3] =", sl)

# Write into a sub-region of a cell:
t.putcellslice("DATA", 2, np.array([[99 + 0j, 98 + 0j]]), blc=[0, 1], trc=[0, 2])
print("after putcellslice:", t.getcell("DATA", 2)[0, :3])

cell DATA[2] shape: (2, 4)
DATA[2][0,1:3] = [[2.+1.j 2.+1.j]]
after putcellslice: [ 2.+1.j 99.+0.j 98.+0.j]


## 5. Column slices

`getcolslice` / `putcolslice` read/write the same cell sub-region across a range of rows.

In [7]:
# First 1x4 row-slice of DATA for all rows
cs = t.getcolslice("DATA", blc=[0, 0], trc=[0, 3])
print("colslice shape:", cs.shape)   # (nrow, 1, 4)

colslice shape: (5, 1, 4)


## 6. Variable-shaped columns

`DATA` was declared with `ndim=2` but no fixed `shape`, so each row may hold a different
shape. `getvarcol` returns a dict of per-row arrays (since they need not be stackable).

In [8]:
t.putcell("DATA", 4, np.ones((3, 2), dtype=np.complex128))   # different shape in row 4
vc = t.getvarcol("DATA")
keys = list(vc.keys())
print("getvarcol keys:", keys)
print("first row shape:", np.asarray(vc[keys[0]]).shape,
      " last row shape:", np.asarray(vc[keys[-1]]).shape)

getvarcol keys: ['r1', 'r2', 'r3', 'r4', 'r5']
first row shape: (1, 2, 4)  last row shape: (1, 3, 2)


## 7. Keywords

Tables and individual columns carry keyword sets (a record / dict). Use
`putkeyword` / `getkeyword` for one keyword, or `putkeywords` / `getkeywords` for the
whole set. Column keywords are addressed by passing the column name.

In [9]:
# table-level keywords
t.putkeyword("MS_VERSION", 2.0)
t.putkeyword("OBS", {"telescope": "VLA", "project": "demo"})   # nested record
print("table keywords:", t.getkeywords())
print("table keyword names:", t.keywordnames())

# column keywords go through the column accessor (t.col(name))
uvwcol = t.col("UVW")
uvwcol.putkeyword("QuantumUnits", ["m", "m", "m"])
print("UVW column keywords:", uvwcol.getkeywords())

table keywords: {'MS_VERSION': 2.0, 'OBS': {'telescope': 'VLA', 'project': 'demo'}}
table keyword names: ['MS_VERSION', 'OBS']
UVW column keywords: {'QuantumUnits': ['m', 'm', 'm']}


## 8. Adding / removing / renaming columns and rows

In [10]:
# add a column built from an existing column's description
t.addcols(makecoldesc("ANTENNA2", t.getcoldesc("ANTENNA")))
print("after addcols:", t.colnames())

t.renamecol("ANTENNA2", "ANTENNA_B")
print("after renamecol:", "ANTENNA_B" in t.colnames())

t.removecols("ANTENNA_B")
print("after removecols:", "ANTENNA_B" not in t.colnames())

t.addrows(2)
print("nrows after addrows:", t.nrows())
t.removerows([5, 6])
print("nrows after removerows:", t.nrows())

after addcols: ['ANTENNA', 'NAME', 'TIME', 'FLAG_ROW', 'UVW', 'DATA', 'ANTENNA2']
after renamecol: True
after removecols: True
nrows after addrows: 7
nrows after removerows: 5


## 9. `tablecolumn` — the column accessor

`t.col(name)` (or `tablecolumn(t, name)`) gives a sequence-like view of a single column
supporting indexing, slicing, `getcol`/`putcol`, `datatype`, `keywordnames`, and more.

In [11]:
antcol = t.col("ANTENNA")
print("len:", len(antcol), "| antcol[2] =", antcol[2], "| antcol[1:4] =", antcol[1:4])
antcol[0] = 100
print("after antcol[0]=100 ->", t.getcell("ANTENNA", 0))
print("datatype:", antcol.datatype(), "| isscalar:", antcol.isscalar())

len: 5 | antcol[2] = 2 | antcol[1:4] = [1, 2, 3]
after antcol[0]=100 -> 100
datatype: int | isscalar: True


## 10. `tablerow` — row-at-a-time access

`t.row()` returns a `tablerow` whose elements are dicts mapping column name → value.
You can restrict it to a subset of columns and write rows back.

In [12]:
tr = t.row(["ANTENNA", "NAME", "TIME"])
print("row 1:", tr[1])

tr.put(1, {"ANTENNA": 42, "NAME": "ea99", "TIME": 1.5})
print("row 1 after put:", tr[1])

row 1: {'ANTENNA': 1, 'NAME': 'ea02', 'TIME': 1.0}
row 1 after put: {'ANTENNA': 42, 'NAME': 'ea99', 'TIME': 1.5}


## 11. TaQL — the Table Query Language

`taql(command)` runs a query string and returns a (reference) table. The calling table is
referenced as `$t`. `query`, `select`, `sort`, and `calc` are convenience wrappers.

In [13]:
sub = taql("select ANTENNA, NAME, TIME from $t where ANTENNA >= 2")
print("selected rows:", sub.nrows())
print("ANTENNA:", sub.getcol("ANTENNA"))

# convenience methods on the table object
hi = t.query("ANTENNA > 2", columns="ANTENNA,NAME")
print("query() rows:", hi.nrows())

ordered = t.sort("TIME DESC")
print("sorted TIME:", ordered.getcol("TIME"))

# scalar expression with calc()
print("max ANTENNA via calc:", t.calc("max(ANTENNA)"))

selected rows: 5
ANTENNA: [100  42   2   3   4]
query() rows: 4
sorted TIME: [4.  3.  2.  1.5 0. ]
max ANTENNA via calc: [100  42   2   3   4]


## 12. Iterating a table — `iter` / `tableiter`

`t.iter(columns)` yields a sub-table for each distinct value (group) of the given
column(s).

In [14]:
# group rows by FLAG_ROW value
t.putcol("FLAG_ROW", np.array([False, False, True, True, False]))
for i, group in enumerate(t.iter("FLAG_ROW")):
    print(f"group {i}: FLAG_ROW={group.getcell('FLAG_ROW', 0)}  nrows={group.nrows()}")

group 0: FLAG_ROW=False  nrows=3
group 1: FLAG_ROW=True  nrows=2


## 13. Indexed lookups — `index` / `tableindex`

`t.index(columns)` builds an in-memory index for fast row lookup by value.

In [15]:
inx = t.index("ANTENNA")
print("is unique:", inx.isunique())
print("rownr where ANTENNA==3:", inx.rownr(3))
print("rownrs where ANTENNA==42:", inx.rownrs(42))

is unique: True
rownr where ANTENNA==3: 3
rownrs where ANTENNA==42: [1]


## 14. Copy / rename / delete and ASCII import-export

In [16]:
# deep copy, then rename the copy
t.copy("demo_copy.tab", deep=True)
tablerename("demo_copy.tab", "demo_renamed.tab")
print("renamed exists:", tableexists("demo_renamed.tab"))

# write a few columns to ASCII and read them back into a new table
t.toascii("demo.txt", columnnames=["ANTENNA", "TIME"])
ascii_tab = tablefromascii("demo_from_ascii.tab", "demo.txt")
print("from-ascii columns:", ascii_tab.colnames(), "nrows:", ascii_tab.nrows())
ascii_tab.close()

tabledelete("demo_renamed.tab")
print("after delete, exists:", tableexists("demo_renamed.tab"))

renamed exists: True
Input format: [ANTENNA=I, TIME=D]
Successful readonly open of default-locked table demo_from_ascii.tab: 2 columns, 5 rows
from-ascii columns: ['ANTENNA', 'TIME'] nrows: 5
Table demo_renamed.tab has been deleted
after delete, exists: False


## 15. Re-opening tables and the table cache

Open an existing table read-only or read/write with `table(name, readonly=...)`. Always
`close()` tables you open; closing also removes them from casacore's table cache so the
same path can be opened again.

In [17]:
t.flush()
ro = table("demo.tab", readonly=True)
print("reopened read-only, writable =", ro.iswritable(), "nrows =", ro.nrows())
ro.close()

# tablecommand is the general entry point taql() is built on
res = tablecommand("select from demo.tab where TIME > 1.0")
print("tablecommand rows:", res.nrows())
res.close()

Successful readonly open of default-locked table demo.tab: 6 columns, 5 rows
reopened read-only, writable = True nrows = 5
tablecommand rows: 4


## Cleanup

In [18]:
t.close()
sub.close(); hi.close(); ordered.close()
os.chdir("/")
shutil.rmtree(workdir, ignore_errors=True)
print("done — scratch directory removed")

done — scratch directory removed


---

### Notes

* **Zero-copy.** `getcol`/`getcellslice`/`getcolslice` return numpy views over casacore
  storage; `getcolnp`/`getcellnp` fill a caller-provided array in place. Writes share the
  numpy buffer when its dtype already matches the column.
* **Axis order.** casacore is column-major; numpy is row-major. The bindings reverse axes
  (and slice corners / shapes) at the boundary, exactly like `python-casacore`, so you
  always work in numpy's row-major convention.
* **No MeasurementSet helpers.** `default_ms`, `msconcat`, `addImagingColumns`, … live in
  casacore's `ms` module, which this tables-only package does not bundle.